## Retrievers 
Retrievers are components (runnables) in langchain, that take user query --> search through data --> Fetch relative documents in response

- There are many types of Retrievers
- Retrievers are Runnables


How is it different from vector store?   
A vector store is just one possible data source through which Retriever can search.     
Vector store can be turned into retiever btw  
There can be more like - SQL queries, Hybrid Search, APIs, etc along with vectore store.

Types of Retrievers:   
These can be classified based on two things - Data Source, and Search methodology
- Based on Data Source:
    - some examples are - Vector Store retiever, Wikipedia retriever etc

- Based on Search methodology: 
    - some examples are - MMR, Multi-query retriever, contextual compression retiever etc 

### 1. Wikipedia Retriever
Queries wikipedia api to fetch relevant content for users query

We give it a query --> Sends query to Wikipedia API --> retrieves most relevant articles (through keyword matching) --> returns them as langchain document

In [1]:
from langchain_community.retrievers import WikipediaRetriever

In [2]:
query = "History of Delhi Metro"

retriver = WikipediaRetriever(
    top_k_results=1,
    lang='en',
    load_all_available_meta=False
)

docs = retriver.invoke(query)

In [3]:
for idx, doc in enumerate(docs):
    print("----- Result ", idx, " -----" )
    print(doc)
    print("\n\n\n")

----- Result  0  -----
page_content='The Delhi Metro is a rapid transit system that serves Delhi and the adjoining satellite cities of Faridabad, Gurugram, Ghaziabad, Noida, Bahadurgarh, and Ballabhgarh in the National Capital Region of India. The system consists of 10 colour-coded lines serving 271 stations, with a total length of 374.466 km (232.682 mi). It is India's largest and busiest metro rail system. The metro has a mix of underground, at-grade, and elevated stations using broad-gauge and standard-gauge tracks. The metro makes over 4,300 trips daily.
Construction began in 1998, and the first elevated section (Shahdara to Tis Hazari) on the Red Line opened on 25 December 2002. The first underground section (Vishwa Vidyalaya – Kashmere Gate) on the Yellow Line opened on 20 December 2004. The network was developed in phases. Phase I was completed by 2006, followed by Phase II in 2011. Phase III was mostly complete in 2021, except for a small extension of the Airport Line which ope

### 2. Vector Store Retriever 
Here we use Vector stores ar retriver  
how is it different from .similarity_search()?   
- Retriever is a runnable, hence can be integrated in chains easily
- .similarity_search() gives you fixed technique for searching. With help me of retievers we can implement many types of searches in vector stores

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
from langchain_core.documents import Document

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-v2"
)

documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model)

In [6]:
retriever = vector_store.as_retriever()

results = retriever.invoke(input="what does langchain do?", k=2)

In [7]:
for idx, doc in enumerate(results):
    print("----- Result ", idx, " -----" )
    print(doc)
    print("\n\n")

----- Result  0  -----
page_content='LangChain helps developers build LLM applications easily.'



----- Result  1  -----
page_content='Chroma is a vector database optimized for LLM-based search.'





### MMR - Maximum Marginal Relevance 
A search technique which solves a question - "_How can we pick results that are relevant to the query but also different from each other?_".

Consider a senerio where you get results but almost all of them says the same thing - that is wastage of resources right, hence sometimes we want results to have diversity. MMR helps there.

"MMR is a information retrieval algorithm designed to reduce redundancy in the retrieved results while also maintaining high relevance to query"

In [8]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [ ]:
vector_store = Chroma.from_documents(
    embedding=embedding_model,
    documents=docs
)

In [22]:
retriever = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs=({'k':3,
    'lambda_mult' : 1})  # 1-->gives highly similar results , 0--> gives highly diverse results
)

In [23]:
results = retriever.invoke("What is langchain?")
for idx, doc in enumerate(results):
    print("----- Result ", idx, " -----" )
    print(doc)
    print("\n\n")

----- Result  0  -----
page_content='LangChain supports Chroma, FAISS, Pinecone, and more.'



----- Result  1  -----
page_content='LangChain is used to build LLM based applications.'



----- Result  2  -----
page_content='LangChain helps developers build LLM applications easily.'





In [25]:
retriever = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs=({'k':3,
    'lambda_mult' : 0})  # 1-->gives highly similar results , 0--> gives highly diverse results
)
results = retriever.invoke("What is langchain?")
for idx, doc in enumerate(results):
    print("----- Result ", idx, " -----" )
    print(doc)
    print("\n\n")

----- Result  0  -----
page_content='LangChain supports Chroma, FAISS, Pinecone, and more.'



----- Result  1  -----
page_content='Embeddings convert text into high-dimensional vectors.'



----- Result  2  -----
page_content='MMR helps you get diverse results when doing similarity search.'





Some other retievers are 
- Multi query retriever
- Contextual compression retriver 

There are many more Retrievers present in langchain so yeahh keep learning as you build. 

[Retrivers](https://docs.langchain.com/oss/python/integrations/retrievers/index#retriever-integrations)